# NB_OpenAI_Codex_BronzeToSilver

**Source:** `AI_Medallion.Bronze.openai_usage`
**Reads:** `okta_user_for_ai` (`TABLE_OKTA_SILVER`)
**Targets:** `codex_usage` (matched OKTA), `codex_usage_matching_log` (unmatched after priorities 1–5)
**Run after:** `NB_OktaUserforAI_Bronze_To_Silver.ipynb`

Processing filter: `okta_user_id IS NOT NULL` → silver; `IS NULL` → matching log. Full overwrite both targets.

In [ ]:
import os
from pathlib import Path
_cwd = Path.cwd()
if not (_cwd / "Common" / "bootstrap.py").is_file():
    _repo = _cwd.parent
    if (_repo / "Common" / "bootstrap.py").is_file():
        os.chdir(_repo)
REBUILD_MEDALLION_CACHE = False
os.environ.setdefault("MEDALLION_DEBUG_READ", "1")
from Common.io import sync_rebuild_medallion_cache_env

sync_rebuild_medallion_cache_env(REBUILD_MEDALLION_CACHE)


In [ ]:
%run ./Common/bootstrap


In [ ]:
bronze = read_table(TABLE_OPENAI_USAGE_BRONZE)
okta = read_table(TABLE_OKTA_SILVER)

codex_silver, codex_matching_log = build_codex_usage_silver(bronze, okta)

write_full_table(codex_silver, TABLE_CODEX_USAGE_SILVER)
write_full_table(codex_matching_log, TABLE_CODEX_USAGE_MATCHING_LOG_SILVER)

show_sample(codex_silver.limit(10))
if codex_matching_log.head(1):
    show_sample(codex_matching_log.limit(10))
